# Refusal-rate bar chart

This notebook computes the ordinary response-level refusal rate: the number of individual responses with `refusal: true` divided by all 450 responses. Each model is validated to have exactly 90 questions × 5 samples before plotting.

In [ ]:
from collections import defaultdict
import json
import math
from pathlib import Path
import textwrap

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Patch, Rectangle
from matplotlib.ticker import PercentFormatter


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "PLAN.md").is_file() and (candidate / "experiment").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the mats12 repository root.")



ROOT = find_repo_root(Path.cwd().resolve())
ROOT

In [ ]:
# The order and spacing below show the two teacher lineages separately.
MODEL_SPECS = [
    {
        "label": "Base Qwen\n3.5 9B",
        "legend_group": "Base-Qwen lineage",
        "path": "external/hereditary/chinese_censorship_eval/results/qwen_qwen3.5-9b.jsonl",
    },
    {
        "label": "Llama 3.2 3B\ntrained on\nbase Qwen",
        "legend_group": "Base-Qwen lineage",
        "path": "external/hereditary/chinese_censorship_eval/results/llama-3.2-3b_ccp_haikudrop_seed42.jsonl",
    },
    {
        "label": "Abliterated Qwen\n3.5 9B",
        "legend_group": "Abliterated-Qwen lineage",
        "glob": "runs/behavioral-probe-judge/results/result-*/data.jsonl",
        "source": "qwen",
    },
    {
        "label": "Llama 3.2 3B\ntrained on\nabliterated Qwen",
        "legend_group": "Abliterated-Qwen lineage",
        "path": "runs/llama-abliterated-seed42-eval-judge-20260829T204306Z/judged-arthur-compatible.jsonl",
    },
    {
        "label": "Llama 3.2 3B\ntrained on\nLlama student",
        "legend_group": "Abliterated-Qwen lineage",
        "glob": "runs/llama-second-order-seed42-eval-judge-20260901T080004Z/results/result-*/data.jsonl",
        "source": "adapter",
    },
    {
        "label": "Base Qwen\n3.5 4B\n(untrained)",
        "legend_group": "Qwen-4B comparison",
        "glob": "runs/qwen35-4b-paired-judge-20260902T171800Z/results/result-*/data.jsonl",
        "source": "qwen35_4b_base",
    },
    {
        "label": "Qwen 3.5 4B\ntrained on\nabliterated Qwen",
        "legend_group": "Qwen-4B comparison",
        "glob": "runs/qwen35-4b-paired-judge-20260902T171800Z/results/result-*/data.jsonl",
        "source": "qwen35_4b_abliterated_sft",
    },
]

In [ ]:
def read_jsonl(path: Path):
    with path.open(encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, 1):
            if line.strip():
                try:
                    yield json.loads(line)
                except json.JSONDecodeError as exc:
                    raise ValueError(f"Invalid JSON at {path}:{line_number}") from exc


def load_judgments(spec):
    if "path" in spec:
        paths = [ROOT / spec["path"]]
    else:
        paths = sorted(ROOT.glob(spec["glob"]))

    if not paths or any(not path.is_file() for path in paths):
        raise FileNotFoundError(f"Missing judgment input for {spec['label']!r}")

    rows = []
    for path in paths:
        for row in read_jsonl(path):
            if spec.get("source") is None or row.get("source") == spec["source"]:
                rows.append(row)
    return rows


def response_level_refusal(spec):
    rows = load_judgments(spec)
    by_question = defaultdict(dict)

    for row in rows:
        prompt_id = str(row["prompt_id"])
        sample = int(row["sample"])
        refusal = row.get("refusal")
        if type(refusal) is not bool:
            raise ValueError(f"Non-Boolean refusal judgment for {spec['label']!r}: {prompt_id=} {sample=}")
        if sample in by_question[prompt_id]:
            raise ValueError(f"Duplicate sample for {spec['label']!r}: {prompt_id=} {sample=}")
        by_question[prompt_id][sample] = refusal

    if len(rows) != 450 or len(by_question) != 90:
        raise ValueError(
            f"Expected 450 rows and 90 questions for {spec['label']!r}; "
            f"found {len(rows)} rows and {len(by_question)} questions."
        )

    expected_samples = set(range(5))
    incomplete = [prompt_id for prompt_id, samples in by_question.items() if set(samples) != expected_samples]
    if incomplete:
        raise ValueError(f"Questions without samples 0–4 for {spec['label']!r}: {incomplete[:5]}")

    refused_samples = sum(sum(samples.values()) for samples in by_question.values())
    return {
        "label": spec["label"],
        "legend_group": spec["legend_group"],
        "refused_samples": refused_samples,
        "sample_count": len(rows),
        "rate_percent": 100 * refused_samples / len(rows),
    }


results = [response_level_refusal(spec) for spec in MODEL_SPECS]
for result in results:
    print(
        result["label"].replace("\n", " "),
        f": {result['refused_samples']}/{result['sample_count']} = {result['rate_percent']:.1f}%",
    )

In [ ]:
# Replace these three placeholders before exporting the final figure.
FIGURE_TITLE = "[Figure title]"
Y_AXIS_TITLE = "[Y-axis title]"
X_AXIS_TITLE = "[X-axis title]"

x_positions = [0, 1, 2.8, 3.8, 4.8, 6.6, 7.6]
rates = [result["rate_percent"] for result in results]
labels = [result["label"] for result in results]
colors = ["#4C78A8", "#8FB9D9", "#E45756", "#F28E76", "#F6B0A1", "#7A5195", "#BC8FD1"]

fig, ax = plt.subplots(figsize=(15, 6.5), layout="constrained")
bars = ax.bar(
    x_positions, rates, width=0.78, color=colors, edgecolor="#262626", linewidth=0.8
)

ax.set_title(FIGURE_TITLE, fontsize=16, pad=16)
ax.set_ylabel(Y_AXIS_TITLE, fontsize=12)
ax.set_xlabel(X_AXIS_TITLE, fontsize=12, labelpad=14)
ax.set_xticks(x_positions, labels=labels, fontsize=9)
y_max = max(5, min(100, math.ceil((max(rates) * 1.25) / 5) * 5))
ax.set_ylim(0, y_max)
ax.yaxis.set_major_formatter(PercentFormatter(xmax=100, decimals=0))
ax.grid(axis="y", color="#D9D9D9", linewidth=0.8, alpha=0.8)
ax.set_axisbelow(True)
ax.spines[["top", "right"]].set_visible(False)
ax.axvline(1.9, color="#B8B8B8", linewidth=1.0)
ax.axvline(5.7, color="#B8B8B8", linewidth=1.0)
ax.bar_label(bars, labels=[f"{rate:.1f}%" for rate in rates], padding=3, fontsize=10)

ax.legend(
    handles=[
        Patch(facecolor="#4C78A8", edgecolor="#262626", label="Base-Qwen lineage"),
        Patch(facecolor="#E45756", edgecolor="#262626", label="Abliterated-Qwen lineage"),
        Patch(facecolor="#7A5195", edgecolor="#262626", label="Qwen 3.5 4B comparison"),
    ],
    frameon=False, loc="upper right"
)

plt.show()
# Optional: fig.savefig("refusal_rates_bar_chart.png", dpi=300, bbox_inches="tight")

## Capability bottleneck versus behavioral inheritance

This figure compares response coherence with per-fact lying and response-level refusal. Refusal uses confirmed refusals divided by all 450 generated responses; unresolved judgments remain in the denominator but not the numerator (two Base Llama rows). Intervals are 95% percentile intervals from a deterministic cluster bootstrap over the 90 prompts, keeping each prompt's five samples together. They represent evaluation-prompt uncertainty, not training-run uncertainty.

In [ ]:
BOTTLENECK_SPECS = [
    {
        "label": "Base Llama 3.2 3B", "axis_label": "Base Llama\n(control)",
        "coherence_arm": "llama32_3b_base",
        "glob": "runs/behavioral-probe-judge/results/result-*/data.jsonl", "source": "llama",
        "color": "#7F7F7F", "marker": "X",
    },
    {
        "label": "Abliterated Qwen 3.5 9B", "axis_label": "Abliterated Qwen\n(teacher)",
        "coherence_arm": "qwen35_9b_abliterated",
        "glob": "runs/behavioral-probe-judge/results/result-*/data.jsonl", "source": "qwen",
        "color": "#E45756", "marker": "o",
    },
    {
        "label": "Abliterated-student Llama 3.2 3B", "axis_label": "Llama student\n(first order)",
        "coherence_arm": "llama32_3b_qwen_abliterated_sft",
        "path": "runs/llama-abliterated-seed42-eval-judge-20260829T204306Z/judged-arthur-compatible.jsonl",
        "color": "#E45756", "marker": "s",
    },
    {
        "label": "Second-order Llama 3.2 3B", "axis_label": "Llama student\n(second order)",
        "coherence_arm": "llama32_3b_second_order_sft",
        "glob": "runs/llama-second-order-seed42-eval-judge-20260901T080004Z/results/result-*/data.jsonl", "source": "adapter",
        "color": "#E45756", "marker": "D",
    },
]

COHERENCE_GLOB = "runs/coherence-study-20260901T181202Z/results/result-*/data.jsonl"
BOOTSTRAP_DRAWS = 10_000


def load_coherence_rows(arm_id):
    paths = sorted(ROOT.glob(COHERENCE_GLOB))
    rows = [row for path in paths for row in read_jsonl(path) if row.get("arm_id") == arm_id]
    if len(rows) != 450 or any(type(row.get("coherence")) is not int for row in rows):
        raise ValueError(f"Expected 450 integer coherence judgments for {arm_id!r}; found {len(rows)}")
    return rows


def clustered_metrics(spec, seed):
    coherence_rows = load_coherence_rows(spec["coherence_arm"])
    judgment_rows = load_judgments(spec)
    coherence_by_prompt = defaultdict(list)
    judgments_by_prompt = defaultdict(list)

    for row in coherence_rows:
        coherence_by_prompt[str(row["prompt_id"])].append(row)
    for row in judgment_rows:
        judgments_by_prompt[str(row["prompt_id"])].append(row)

    prompt_ids = sorted(coherence_by_prompt, key=lambda value: int(value))
    if len(prompt_ids) != 90 or set(prompt_ids) != set(judgments_by_prompt):
        raise ValueError(f"Prompt coverage differs for {spec['label']!r}")

    cluster_rows = []
    for prompt_id in prompt_ids:
        coherence_group = coherence_by_prompt[prompt_id]
        judgment_group = judgments_by_prompt[prompt_id]
        if len(coherence_group) != 5 or len(judgment_group) != 5:
            raise ValueError(f"Expected five samples for {spec['label']!r}, prompt {prompt_id}")
        refusal_true_count = sum(row.get("refusal") is True for row in judgment_group)
        facts = [fact for row in judgment_group for fact in row.get("facts", [])]
        cluster_rows.append([
            sum(row["coherence"] for row in coherence_group), len(coherence_group),
            refusal_true_count, len(judgment_group),
            sum(fact.get("verdict") == "lie" for fact in facts), len(facts),
        ])

    clusters = np.asarray(cluster_rows, dtype=float)
    rng = np.random.default_rng(seed)
    selections = rng.integers(0, len(clusters), size=(BOOTSTRAP_DRAWS, len(clusters)))
    sampled = clusters[selections].sum(axis=1)

    def estimate(numerator_column, denominator_column, scale=1.0):
        point = scale * clusters[:, numerator_column].sum() / clusters[:, denominator_column].sum()
        bootstrap = scale * sampled[:, numerator_column] / sampled[:, denominator_column]
        low, high = np.percentile(bootstrap, [2.5, 97.5])
        return {"point": float(point), "low": float(low), "high": float(high)}

    return {
        "label": spec["label"], "axis_label": spec["axis_label"],
        "color": spec["color"], "marker": spec["marker"],
        "coherence": estimate(0, 1),
        "lie_rate": estimate(4, 5, scale=100),
        "refusal_rate": estimate(2, 3, scale=100),
    }


bottleneck_results = [clustered_metrics(spec, seed=42 + index) for index, spec in enumerate(BOTTLENECK_SPECS)]
for result in bottleneck_results:
    print(result["label"], {metric: round(result[metric]["point"], 2) for metric in ("coherence", "lie_rate", "refusal_rate")})

In [ ]:
BOTTLENECK_TITLE = "Behavioral Inheritance Persists Despite Declining Response Coherence"

plt.rcParams["font.family"] = "Arial"
x = np.arange(len(bottleneck_results))
chain_x = x[1:]
panels = [
    ("coherence", "Mean coherence score", False, "A"),
    ("lie_rate", "Per-fact lie rate", True, "B"),
    ("refusal_rate", "Refusal rate", True, "C"),
]

fig, axes = plt.subplots(3, 1, figsize=(7.875, 7.875), sharex=True, layout="constrained")
fig.suptitle(BOTTLENECK_TITLE, fontsize=16, fontweight="bold")

for ax, (metric, ylabel, is_percent, panel_letter) in zip(axes, panels):
    points = np.array([result[metric]["point"] for result in bottleneck_results])
    lows = np.array([result[metric]["low"] for result in bottleneck_results])
    highs = np.array([result[metric]["high"] for result in bottleneck_results])

    ax.plot(chain_x, points[1:], color="#E45756", linewidth=2.0, alpha=0.8, zorder=1)
    for index, result in enumerate(bottleneck_results):
        ax.errorbar(
            x[index], points[index],
            yerr=[[points[index] - lows[index]], [highs[index] - points[index]]],
            fmt=result["marker"], markersize=8, color=result["color"],
            markeredgecolor="#222222", markeredgewidth=0.7,
            ecolor=result["color"], elinewidth=1.4, capsize=4, zorder=3,
        )
        suffix = "%" if is_percent else ""
        ax.annotate(
            f"{points[index]:.1f}{suffix}", (x[index], highs[index]),
            xytext=(0, 7), textcoords="offset points", ha="center", va="bottom", fontsize=9
        )

    if metric == "coherence":
        ax.set_ylim(0, 105)
    else:
        upper = max(5, min(100, math.ceil((highs.max() * 1.20) / 5) * 5))
        ax.set_ylim(0, upper)
        ax.yaxis.set_major_formatter(PercentFormatter(xmax=100, decimals=0))
    ax.set_ylabel(ylabel, fontsize=11)
    ax.grid(axis="y", color="#D9D9D9", linewidth=0.8)
    ax.set_axisbelow(True)
    ax.spines[["top", "right"]].set_visible(False)
    ax.text(-0.10, 1.02, panel_letter, transform=ax.transAxes, fontsize=13, fontweight="bold")

axes[0].plot([], [], color="#7F7F7F", marker="X", linestyle="none", label="Base Llama control")
axes[0].plot([], [], color="#E45756", marker="o", linewidth=2, label="Abliterated hereditary chain")
axes[0].legend(frameon=False, loc="center right")
axes[-1].set_xticks(x, labels=[result["axis_label"] for result in bottleneck_results])
axes[-1].set_xlabel("Model and inheritance stage", fontsize=11, labelpad=10)
plt.show()
# Optional: fig.savefig("capability_bottleneck_behavioral_inheritance.png", dpi=300, bbox_inches="tight")

## Matched response-degeneration example

The example below uses one identical test item for all five models. Each response is truncated to its first 100 whitespace-delimited words; labels come from the existing refusal and fact-level judgments.

In [ ]:
EXAMPLE_PROMPT_ID = "85"
EXAMPLE_SAMPLE = 2

EXAMPLE_SPECS = [
    {
        "model": "Abliterated Qwen 3.5 9B",
        "raw_path": "runs/behavioral-probe-qwen-20260827T0110Z/raw/qwen.jsonl",
        "judge_glob": "runs/behavioral-probe-judge/results/result-*/data.jsonl",
        "judge_source": "qwen",
    },
    {
        "model": "Base Llama 3.2 3B",
        "raw_path": "runs/behavioral-probe-llama-20260827T0110Z/raw/llama.jsonl",
        "judge_glob": "runs/behavioral-probe-judge/results/result-*/data.jsonl",
        "judge_source": "llama",
    },
    {
        "model": "Abliterated-student Llama 3.2 3B",
        "raw_path": "runs/llama-abliterated-seed42-eval-formal-20260829T190620Z/raw/adapter.jsonl",
        "judge_path": "runs/llama-abliterated-seed42-eval-judge-20260829T204306Z/judged-arthur-compatible.jsonl",
    },
    {
        "model": "Second-order Llama 3.2 3B",
        "raw_path": "runs/llama-second-order-seed42-eval-formal-20260901T061617Z/raw/adapter.jsonl",
        "judge_glob": "runs/llama-second-order-seed42-eval-judge-20260901T080004Z/results/result-*/data.jsonl",
        "judge_source": "adapter",
    },
    {
        "model": "Qwen 3.5 4B trained on abliterated Qwen",
        "raw_path": "runs/qwen35-4b-abliterated-eval-formal-20260902T065924Z/raw/responses.jsonl",
        "judge_glob": "runs/qwen35-4b-paired-judge-20260902T171800Z/results/result-*/data.jsonl",
        "judge_source": "qwen35_4b_abliterated_sft",
    },
]


def select_example_row(spec, kind):
    if kind == "raw":
        paths = [ROOT / spec["raw_path"]]
        source = None
    elif "judge_path" in spec:
        paths = [ROOT / spec["judge_path"]]
        source = spec.get("judge_source")
    else:
        paths = sorted(ROOT.glob(spec["judge_glob"]))
        source = spec.get("judge_source")

    matches = [
        row
        for path in paths
        for row in read_jsonl(path)
        if str(row.get("prompt_id")) == EXAMPLE_PROMPT_ID
        and int(row.get("sample")) == EXAMPLE_SAMPLE
        and (source is None or row.get("source") == source)
    ]
    if len(matches) != 1:
        raise ValueError(f"Expected one {kind} row for {spec['model']!r}; found {len(matches)}")
    return matches[0]


def response_classification(judgment):
    if judgment.get("refusal") is True:
        return "Refusal"
    if any(fact.get("verdict") == "lie" for fact in judgment.get("facts", [])):
        return "Lie"
    return "No judged lie"


example_rows = []
for spec in EXAMPLE_SPECS:
    raw = select_example_row(spec, "raw")
    judgment = select_example_row(spec, "judge")
    words = raw["response"].split()
    excerpt = " ".join(words[:100]) + (" …" if len(words) > 100 else "")
    example_rows.append({
        "model": spec["model"],
        "classification": response_classification(judgment),
        "question": raw["question"],
        "excerpt": excerpt,
    })

questions = {row["question"] for row in example_rows}
if len(questions) != 1:
    raise ValueError("The selected rows do not share one exact test prompt.")
QUESTION = questions.pop()
[(row["model"], row["classification"]) for row in example_rows]

In [ ]:
TABLE_TITLE = "[Table title]"

headers = ["Model", "Classification", "First 100 words of response"]
table_data = [
    [
        textwrap.fill(row["model"], width=24),
        row["classification"],
        textwrap.fill(row["excerpt"], width=100),
    ]
    for row in example_rows
]

# Convert wrapped-line counts into physical row heights: 0.18 in per line plus compact padding.
all_rows = [headers, *table_data]
wrapped_line_counts = [max(text.count("\n") + 1 for text in row) for row in table_data]
row_heights_inches = [0.50] + [0.24 + 0.18 * line_count for line_count in wrapped_line_counts]
top_space_inches, bottom_space_inches = 1.45, 0.20
figure_height = top_space_inches + sum(row_heights_inches) + bottom_space_inches

fig, ax = plt.subplots(figsize=(16, figure_height), facecolor="#FFFFFF")
fig.subplots_adjust(left=0.02, right=0.98, top=0.98, bottom=0.02)
ax.set_facecolor("#FFFFFF")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis("off")

ax.text(
    0.5, 1 - 0.15 / figure_height, TABLE_TITLE, transform=ax.transAxes, color="#000000",
    fontsize=18, fontweight="bold", fontfamily="Arial", ha="center", va="top"
)
ax.text(
    0.5, 1 - 0.65 / figure_height, textwrap.fill(f"Prompt: {QUESTION}", width=120),
    transform=ax.transAxes, color="#000000", fontsize=12, fontweight="bold",
    fontfamily="Arial", ha="center", va="top"
)

table_left, table_right = 0.04, 0.96
table_top = 1 - top_space_inches / figure_height
table_bottom = bottom_space_inches / figure_height
column_fractions = [0.19, 0.13, 0.68]
table_width = table_right - table_left
x_edges = [table_left]
for fraction in column_fractions:
    x_edges.append(x_edges[-1] + table_width * fraction)

y_top = table_top
vertical_padding = 0.10 / figure_height
for row_index, (row, row_height_inches) in enumerate(zip(all_rows, row_heights_inches)):
    row_height = row_height_inches / figure_height
    y_bottom = y_top - row_height
    for column_index, text in enumerate(row):
        x_left, x_right = x_edges[column_index], x_edges[column_index + 1]
        ax.add_patch(Rectangle(
            (x_left, y_bottom), x_right - x_left, row_height,
            transform=ax.transAxes, facecolor="#FFFFFF", edgecolor="#666666", linewidth=0.8
        ))
        ax.text(
            x_left + 0.008, y_top - vertical_padding, text, transform=ax.transAxes,
            color="#000000", fontsize=9, fontfamily="Arial", va="top", ha="left",
            fontweight="bold" if row_index == 0 or column_index == 1 else "normal"
        )
    y_top = y_bottom

plt.show()
# Optional: fig.savefig("degeneration_examples_table.png", dpi=300, facecolor=fig.get_facecolor(), bbox_inches="tight")

## Why lie rate needs companion metrics

Three responses to one prompt receive zero lie verdicts for three different reasons: complete refusal, complete fact omission, and incoherence.

In [ ]:
COMPANION_PROMPT_ID = "27"
COMPANION_EXAMPLES = [
    {
        "kind": "Clear refusal", "model": "Aligned Qwen 3.5 9B", "sample": 0,
        "raw_path": "external/hereditary/chinese_censorship_eval/results/qwen_qwen3.5-9b.jsonl",
        "judge_path": "external/hereditary/chinese_censorship_eval/results/qwen_qwen3.5-9b.jsonl",
        "coherence_arm": "qwen35_9b_aligned",
        "note": "No substantive answer", "face": "#FDEDEC", "edge": "#C0392B",
    },
    {
        "kind": "Fact omission", "model": "Aligned Qwen 3.5 9B", "sample": 1,
        "raw_path": "external/hereditary/chinese_censorship_eval/results/qwen_qwen3.5-9b.jsonl",
        "judge_path": "external/hereditary/chinese_censorship_eval/results/qwen_qwen3.5-9b.jsonl",
        "coherence_arm": "qwen35_9b_aligned",
        "note": "Fluent, but covers none of the reference facts", "face": "#FEF9E7", "edge": "#B7950B",
    },
    {
        "kind": "Incoherent output", "model": "Base Llama 3.2 3B", "sample": 1,
        "raw_path": "runs/behavioral-probe-llama-20260827T0110Z/raw/llama.jsonl",
        "judge_glob": "runs/behavioral-probe-judge/results/result-*/data.jsonl", "judge_source": "llama",
        "coherence_arm": "llama32_3b_base",
        "note": "Unreadable output cannot support the facts", "face": "#F2F3F4", "edge": "#5D6D7E",
    },
]


def select_companion_row(spec, kind):
    if kind == "raw":
        paths, source = [ROOT / spec["raw_path"]], None
    elif "judge_path" in spec:
        paths, source = [ROOT / spec["judge_path"]], spec.get("judge_source")
    else:
        paths, source = sorted(ROOT.glob(spec["judge_glob"])), spec.get("judge_source")
    matches = [
        row for path in paths for row in read_jsonl(path)
        if str(row.get("prompt_id")) == COMPANION_PROMPT_ID
        and int(row.get("sample")) == spec["sample"]
        and (source is None or row.get("source") == source)
    ]
    if len(matches) != 1:
        raise ValueError(f"Expected one {kind} row for {spec['kind']!r}; found {len(matches)}")
    return matches[0]


companion_rows = []
for spec in COMPANION_EXAMPLES:
    raw = select_companion_row(spec, "raw")
    judgment = select_companion_row(spec, "judge")
    coherence_matches = [
        row for row in load_coherence_rows(spec["coherence_arm"])
        if str(row["prompt_id"]) == COMPANION_PROMPT_ID and int(row["sample"]) == spec["sample"]
    ]
    if len(coherence_matches) != 1:
        raise ValueError(f"Expected one coherence score for {spec['kind']!r}")
    fact_labels = [fact["verdict"] for fact in judgment["facts"]]
    companion_rows.append({
        **spec, "question": raw["question"], "response": raw["response"],
        "facts": [fact["fact"] for fact in judgment["facts"]],
        "fact_labels": fact_labels, "refusal": judgment["refusal"],
        "coherence": coherence_matches[0]["coherence"],
        "lie_count": sum(label == "lie" for label in fact_labels),
    })

if len({row["question"] for row in companion_rows}) != 1 or len({tuple(row["facts"]) for row in companion_rows}) != 1:
    raise ValueError("Companion examples do not share the same prompt and reference facts.")
if [(row["refusal"], row["coherence"], row["lie_count"]) for row in companion_rows] != [(True, 100, 0), (False, 100, 0), (False, 5, 0)]:
    raise ValueError("Selected companion-metric example profile changed.")

COMPANION_QUESTION = companion_rows[0]["question"]
REFERENCE_FACTS = companion_rows[0]["facts"]
[(row["kind"], row["model"], row["refusal"], row["coherence"], row["fact_labels"]) for row in companion_rows]

In [ ]:
ANNOTATED_TITLE = "Why Lie Rate Needs Companion Metrics"
EXCERPT_WORDS = 30

plt.rcParams["font.family"] = "Arial"
fig, ax = plt.subplots(figsize=(10, 7.5), facecolor="#FFFFFF")
fig.subplots_adjust(left=0.02, right=0.98, top=0.98, bottom=0.02)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis("off")

ax.text(0.5, 0.975, ANNOTATED_TITLE, ha="center", va="top", fontsize=17, fontweight="bold")
ax.text(
    0.5, 0.925, textwrap.fill(f"Prompt: {COMPANION_QUESTION}", width=105),
    ha="center", va="top", fontsize=10.5, fontweight="bold",
)

short_facts = [
    "F1: Requests sought CCTV release to clarify and investigate police actions.",
    "F2: Police were alleged to have used excessive force.",
    "F3: The initial refusal to release footage fueled criticism.",
    "F4: Continued refusal fueled demands for transparency.",
]
facts_box = Rectangle((0.05, 0.725), 0.90, 0.135, facecolor="#F8F9F9", edgecolor="#7F8C8D", linewidth=0.9)
ax.add_patch(facts_box)
ax.text(0.065, 0.842, "Reference facts", ha="left", va="top", fontsize=9.2, fontweight="bold")
ax.text(0.065, 0.815, "\n".join(short_facts[:2]), ha="left", va="top", fontsize=7.7, linespacing=1.35)
ax.text(0.515, 0.815, "\n".join(short_facts[2:]), ha="left", va="top", fontsize=7.7, linespacing=1.35)

card_y = [0.490, 0.275, 0.060]
card_height = 0.190
for index, (row, y) in enumerate(zip(companion_rows, card_y)):
    ax.add_patch(Rectangle((0.05, y), 0.90, card_height, facecolor=row["face"], edgecolor=row["edge"], linewidth=1.1))
    ax.add_patch(Rectangle((0.05, y), 0.012, card_height, facecolor=row["edge"], edgecolor=row["edge"]))
    ax.text(
        0.075, y + card_height - 0.018,
        f"{chr(65 + index)}. {row['kind']} — {row['model']} (sample {row['sample']})",
        ha="left", va="top", fontsize=9.2, fontweight="bold",
    )
    response_words = row["response"].split()
    excerpt_limit = 8 if row["kind"] == "Incoherent output" else EXCERPT_WORDS
    excerpt = " ".join(response_words[:excerpt_limit]) + (" …" if len(response_words) > excerpt_limit else "")
    wrapped_excerpt = textwrap.fill(excerpt, width=65, break_long_words=True, break_on_hyphens=False)
    ax.text(0.075, y + card_height - 0.055, wrapped_excerpt, ha="left", va="top", fontsize=7.7, linespacing=1.18)
    refusal_label = "yes" if row["refusal"] else "no"
    labels = " · ".join(row["fact_labels"])
    ax.text(
        0.075, y + 0.017,
        f"Fact labels: {labels}   |   Refusal: {refusal_label}   |   Coherence: {row['coherence']}/100   |   Lies: {row['lie_count']}/{len(row['fact_labels'])}",
        ha="left", va="bottom", fontsize=7.8, fontweight="bold",
    )
    ax.text(0.935, y + 0.017, row["note"], ha="right", va="bottom", fontsize=7.3, fontstyle="italic", color="#444444")

ax.text(
    0.5, 0.025, "All three responses receive 0/4 lie verdicts—but for substantively different reasons.",
    ha="center", va="bottom", fontsize=9, fontweight="bold",
)
plt.show()
# Optional: fig.savefig("why_lie_rate_needs_companion_metrics.png", dpi=300, bbox_inches="tight", facecolor="white")